# 03 – Identification models

**Task:** given a face from the test set, which of the 62 people is it?

We compare three feature sets × three classifiers:

| Features | Classifiers |
|---|---|
| **Eigenfaces** (PCA on raw pixels, the classic 1991 baseline) | **SVM** (RBF kernel) |
| **ArcFace** embeddings | **1-NN** (closest single training image) |
| **Facenet512** embeddings | **Nearest centroid** (closest average face per person, used by the webcam demo) |

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import src...` works from the notebooks folder

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from src.config import SEED
from src.data import load_lfw, split
from src.matching import CentroidMatcher
from src.train import CLASSIFIERS, feature_sets, run_identification

sns.set_theme(style="whitegrid")
images, y, names = load_lfw()
train_idx, test_idx = split(y)
feats = feature_sets(images, train_idx)  # loads cached embeddings
print(f"train: {len(train_idx)} images, test: {len(test_idx)} images, {len(names)} people")

## 1. Comparison table

In [ ]:
results = run_identification(feats, y, train_idx, test_idx)
results.round(3).sort_values("macro_f1", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=results, x="features", y="macro_f1", hue="classifier", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("Macro F1 (test set)")
ax.set_xlabel("")
ax.legend(title="", loc="lower right")
plt.tight_layout()
plt.show()

## 2. Closer look at ArcFace + SVM

Macro F1 weights every person equally, so the weakest people matter as much as George W Bush
with his 530 images.

In [ ]:
MODEL = "ArcFace"
clf = CLASSIFIERS["SVM"]().fit(feats[MODEL][train_idx], y[train_idx])
pred = clf.predict(feats[MODEL][test_idx])

report = pd.DataFrame(classification_report(y[test_idx], pred, target_names=names,
                                            output_dict=True, zero_division=0)).T
print("Weakest 10 people:")
report.iloc[:-3].sort_values("f1-score").head(10).round(2)

In [ ]:
cm = confusion_matrix(y[test_idx], pred, normalize="true")
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(cm, cmap="Blues", square=True, xticklabels=False, yticklabels=False,
            cbar_kws={"label": "Share of true class"}, ax=ax)
ax.set_xlabel("Predicted person")
ax.set_ylabel("True person")
ax.set_title(f"{MODEL} + SVM: confusion matrix (a strong diagonal = good)")
plt.tight_layout()
plt.show()

In [ ]:
wrong = pred != y[test_idx]
wrong_idx, wrong_pred = test_idx[wrong], pred[wrong]
print(f"{wrong.sum()} of {len(test_idx)} test images misclassified")

fig, axes = plt.subplots(2, 6, figsize=(14, 6))
for ax in axes.flat:
    ax.axis("off")
for ax, i, p in zip(axes.flat, wrong_idx, wrong_pred):
    ax.imshow(images[i])
    ax.set_title(f"true: {names[y[i]].split()[-1]}\npred: {names[p].split()[-1]}", fontsize=9)
plt.suptitle("Misclassified test images: what makes them hard?")
plt.tight_layout()
plt.show()

## 3. How many photos per person do we need?

In a real system you might only have 1–3 photos of a new person. We sample *k* training images
per person (5 repetitions per *k*) and classify with nearest centroid.

Note: the Eigenfaces PCA is still fitted on the full training set, which flatters the baseline
slightly.

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for k in [1, 2, 3, 5, 10]:
    for rep in range(5):
        shots = np.concatenate([rng.choice(train_idx[y[train_idx] == c], size=k, replace=False)
                                for c in np.unique(y)])
        for fname, F in feats.items():
            p = CentroidMatcher().fit(F[shots], y[shots]).predict(F[test_idx])
            rows.append({"images per person": k, "features": fname, "accuracy": np.mean(p == y[test_idx])})
curve = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=curve, x="images per person", y="accuracy", hue="features", marker="o",
             errorbar="sd", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Accuracy vs. training images per person (mean ± sd over 5 draws)")
plt.tight_layout()
plt.show()
curve.groupby(["features", "images per person"])["accuracy"].mean().unstack().round(3)

## Observations (fill in after running)

- How big is the gap between Eigenfaces and the deep embeddings?
- Does the classifier matter much once the features are good?
- How many photos per person are enough for ArcFace?
- What do the misclassified images have in common?